# 信号处理基础

射频指纹识别的底层知识是信号处理。本课覆盖:

1. 信号基础: 连续/离散, 采样定理
2. 傅里叶变换: DFT, FFT, 频谱分析
3. IQ信号与调制解调
4. 滤波器设计
5. 信道效应: 衰落、多径、噪声
6. 信号检测与参数估计
7. Python信号处理实战 (scipy.signal)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
plt.rcParams['figure.figsize'] = (12, 4)

## 1. 信号基础

### 连续信号 vs 离散信号

```
连续信号 x(t): 时间连续, 值连续 (物理世界)
离散信号 x[n]: 时间离散, 值连续   (采样后)
数字信号:       时间离散, 值离散   (量化后, 计算机处理)
```

### 采样定理 (Nyquist)

要无失真恢复信号, 采样频率必须 ≥ 2倍信号最高频率:

```
f_s ≥ 2 * f_max
```

不满足则产生混叠 (Aliasing)。

In [ ]:
# 采样定理演示
f_signal = 5  # 信号频率 5Hz
t_continuous = np.linspace(0, 1, 1000)
x_continuous = np.sin(2 * np.pi * f_signal * t_continuous)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 满足采样定理: f_s = 50Hz > 2*5 = 10Hz
fs_ok = 50
t_ok = np.arange(0, 1, 1/fs_ok)
x_ok = np.sin(2 * np.pi * f_signal * t_ok)
axes[0].plot(t_continuous, x_continuous, 'b-', alpha=0.3, label='Continuous')
axes[0].stem(t_ok, x_ok, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled fs={fs_ok}Hz')
axes[0].set_title(f'Proper Sampling (fs={fs_ok}Hz > 2*{f_signal}Hz)')
axes[0].legend(fontsize=8)
axes[0].set_xlabel('Time (s)')

# 临界采样: f_s = 10Hz = 2*5Hz
fs_critical = 10
t_crit = np.arange(0, 1, 1/fs_critical)
x_crit = np.sin(2 * np.pi * f_signal * t_crit)
axes[1].plot(t_continuous, x_continuous, 'b-', alpha=0.3, label='Continuous')
axes[1].stem(t_crit, x_crit, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled fs={fs_critical}Hz')
axes[1].set_title(f'Critical Sampling (fs={fs_critical}Hz = 2*{f_signal}Hz)')
axes[1].legend(fontsize=8)
axes[1].set_xlabel('Time (s)')

# 欠采样: f_s = 7Hz < 2*5 = 10Hz → 混叠!
fs_low = 7
t_low = np.arange(0, 1, 1/fs_low)
x_low = np.sin(2 * np.pi * f_signal * t_low)
f_alias = fs_low - f_signal  # 混叠频率
x_alias = np.sin(2 * np.pi * f_alias * t_continuous)
axes[2].plot(t_continuous, x_continuous, 'b-', alpha=0.3, label=f'Original {f_signal}Hz')
axes[2].plot(t_continuous, x_alias, 'g--', alpha=0.5, label=f'Aliased {f_alias}Hz')
axes[2].stem(t_low, x_low, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled fs={fs_low}Hz')
axes[2].set_title(f'Under Sampling (fs={fs_low}Hz < 2*{f_signal}Hz) → Aliasing!')
axes[2].legend(fontsize=8)
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()
print(f"信号频率: {f_signal}Hz, Nyquist频率: {2*f_signal}Hz")
print(f"欠采样时, {f_signal}Hz信号看起来像{f_alias}Hz → 这就是混叠")

In [ ]:
# 常见信号类型
t = np.linspace(0, 1, 1000)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 正弦波
axes[0, 0].plot(t, np.sin(2*np.pi*5*t))
axes[0, 0].set_title('Sine Wave: sin(2πft)')

# 方波
axes[0, 1].plot(t, signal.square(2*np.pi*5*t))
axes[ 1].set_title('Square Wave')

# 锯齿波
axes[0, 2].plot(t, signal.sawtooth(2*np.pi*5*t))
axes[0, 2].set_title('Sawtooth Wave')

# 高斯脉冲
axes[1, 0].plot(t, signal.gausspulse(t - 0.5, fc=5))
axes[1, 0].set_title('Gaussian Pulse')

# Chirp (线性调频)
axes[1, 1].plot(t, signal.chirp(t, f0=1, f1=20, t1=1, method='linear'))
axes[1, 1].set_title('Chirp (Linear FM)')

# 白噪声
np.random.seed(42)
axes[1, 2].plot(t, np.random.randn(len(t)))
axes[1, 2].set_title('White Noise')

for ax in axes.flatten():
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

---
## 2. 傅里叶变换

### 核心思想

任何信号都可以分解为不同频率的正弦波之和:

```
时域信号 x(t) ←→ 频域信号 X(f)    (傅里叶变换对)

X(f) = ∫ x(t) * e^(-j2πft) dt       (连续傅里叶变换)
X[k] = Σ x[n] * e^(-j2πkn/N)        (离散傅里叶变换, DFT)
```

### 为什么需要频域分析?

- 时域看不出频率成分 → 频域一目了然
- 滤波在频域就是乘法
- 很多信号处理在频域更直观

In [ ]:
# 傅里叶变换: 时域 → 频域
fs = 1000  # 采样率
t = np.arange(0, 1, 1/fs)

# 混合信号: 5Hz + 20Hz + 50Hz
x = 1.0 * np.sin(2*np.pi*5*t) + 0.5 * np.sin(2*np.pi*20*t) + 0.3 * np.sin(2*np.pi*50*t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 时域
axes[0].plot(t[:200], x[:200])
axes[0].set_title('Time Domain')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# 频域
N = len(x)
X = fft(x)
freqs = fftfreq(N, 1/fs)
X_mag = np.abs(X) / N  # 幅度归一化
X_mag = X_mag[:N//2] * 2  # 单边谱
freqs_pos = freqs[:N//2]

axes[1].plot(freqs_pos, X_mag)
axes[1].set_title('Frequency Domain (FFT)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_xlim(0, 80)
axes[1].axvline(5, color='r', linestyle='--', alpha=0.5, label='5Hz')
axes[1].axvline(20, color='g', linestyle='--', alpha=0.5, label='20Hz')
axes[1].axvline(50, color='b', linestyle='--', alpha=0.5, label='50Hz')
axes[1].legend()

plt.tight_layout()
plt.show()
print("时域看不出频率成分, 频域中三个频率分量一目了然")

In [ ]:
# FFT 关键参数
print("=== FFT 关键概念 ===")
print()
print("1. 频率分辨率: Δf = fs / N")
print("   采样率越高或采样点越多, 频率分辨率越高")
print()
print("2. 频率范围: [0, fs/2] (Nyquist频率)")
print("   超过 fs/2 的频率成分会混叠")
print()
print("3. FFT复杂度: O(N log N)")
print("   N最好是2的幂 (256, 512, 1024...)")
print()
print("4. 单边谱 vs 双边谱:")
print("   双边谱: 正负频率对称, 显示 [-fs/2, fs/2]")
print("   单边谱: 只显示 [0, fs/2], 幅度×2 (直流分量除外)")
print()
print("5. 窗函数: 减少频谱泄漏")
print("   矩形窗(默认), 汉宁窗, 汉明窗, 布莱克曼窗")

In [ ]:
# 窗函数与频谱泄漏
fs = 1000
t = np.arange(0, 0.5, 1/fs)
x = np.sin(2*np.pi*50.5*t)  # 频率不是整数周期, 会泄漏

windows = {
    'Rectangular (No Window)': np.ones(len(x)),
    'Hanning': np.hanning(len(x)),
    'Hamming': np.hamming(len(x)),
    'Blackman': np.blackman(len(x)),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (name, win) in zip(axes.flatten(), windows.items()):
    x_windowed = x * win
    X = fft(x_windowed)
    freqs = fftfreq(len(x), 1/fs)
    X_mag = np.abs(X[:len(x)//2]) / len(x) * 2
    freqs_pos = freqs[:len(x)//2]
    
    ax.plot(freqs_pos, 20*np.log10(X_mag + 1e-10))
    ax.set_title(name)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_xlim(30, 70)
    ax.set_ylim(-80, 10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Window Functions and Spectral Leakage', fontsize=13)
plt.tight_layout()
plt.show()

print("矩形窗: 主瓣窄(频率分辨率高), 旁瓣高(泄漏严重)")
print("布莱克曼窗: 主瓣宽(频率分辨率低), 旁瓣低(泄漏小)")
print("实际中常用汉宁窗或汉明窗, 折中分辨率和泄漏")

In [ ]:
# 短时傅里叶变换 (STFT) - 时频分析
fs = 1000
t = np.arange(0, 2, 1/fs)

# 信号频率随时间变化 (Chirp)
x = signal.chirp(t, f0=10, f1=100, t1=2, method='linear')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 时域
axes[0].plot(t, x)
axes[0].set_title('Chirp Signal (Time Domain)')
axes[0].set_xlabel('Time (s)')

# STFT时频图
f, t_stft, Zxx = signal.stft(x, fs=fs, nperseg=128, noverlap=120)
im = axes[1].pcolormesh(t_stft, f, np.abs(Zxx), shading='auto', cmap='viridis')
axes[1].set_title('STFT (Time-Frequency)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim(0, 150)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print("FFT只能看整体频率, STFT可以看到频率随时间的变化")
print("STFT是时频分析的基础, 也是RFFI中时频图输入的来源")

---
## 3. IQ信号与调制解调

### IQ信号

复数基带信号表示:
```
s(t) = I(t) + j*Q(t)

I(t): 同相分量 (In-phase)
Q(t): 正交分量 (Quadrature)
```

### 为什么用IQ?

1. 包含幅度和相位信息
2. 基带表示, 载频已去除
3. 方便数字信号处理

### 调制

将信息(比特)映射到IQ信号:
```
BPSK:  I = ±1,  Q = 0         (2种符号)
QPSK:  I = ±1,  Q = ±1        (4种符号)
16QAM: I,Q各取 ±1, ±3         (16种符号)
```

In [ ]:
# 常见调制方式的星座图
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
np.random.seed(42)
n_symbols = 500

# BPSK
bits = np.random.randint(0, 2, n_symbols)
I_bpsk = 2 * bits - 1
Q_bpsk = np.zeros(n_symbols)
axes[0].scatter(I_bpsk, Q_bpsk, alpha=0.5)
axes[0].set_title('BPSK (2 points)')
axes[0].set_xlabel('I')
axes[0].set_ylabel('Q')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)

# QPSK
bits_i = np.random.randint(0, 2, n_symbols)
bits_q = np.random.randint(0, 2, n_symbols)
I_qpsk = (2 * bits_i - 1) / np.sqrt(2)
Q_qpsk = (2 * bits_q - 1) / np.sqrt(2)
axes[1].scatter(I_qpsk, Q_qpsk, alpha=0.5)
axes[1].set_title('QPSK (4 points)')
axes[1].set_xlabel('I')
axes[1].set_ylabel('Q')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)

# 8PSK
symbols = np.random.randint(0, 8, n_symbols)
angles = symbols * np.pi / 4
I_8psk = np.cos(angles)
Q_8psk = np.sin(angles)
axes[2].scatter(I_8psk, Q_8psk, alpha=0.5)
axes[2].set_title('8PSK (8 points)')
axes[2].set_xlabel('I')
axes[2].set_ylabel('Q')
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(-2, 2)
axes[2].set_ylim(-2, 2)

# 16QAM
bits_i = np.random.randint(0, 4, n_symbols)
bits_q = np.random.randint(0, 4, n_symbols)
I_16qam = (2 * bits_i - 3) / np.sqrt(10)
Q_16qam = (2 * bits_q - 3) / np.sqrt(10)
axes[3].scatter(I_16qam, Q_16qam, alpha=0.5)
axes[3].set_title('16QAM (16 points)')
axes[3].set_xlabel('I')
axes[3].set_ylabel('Q')
axes[3].set_aspect('equal')
axes[3].grid(True, alpha=0.3)
axes[3].set_xlim(-2, 2)
axes[3].set_ylim(-2, 2)

plt.tight_layout()
plt.show()

print("星座图 = IQ平面上的符号点分布")
print("点越多 → 每符号携带比特越多 → 频谱效率高 → 抗噪能力差")

In [ ]:
# 调制与解调过程
fs = 1000
fc = 100  # 载波频率
t = np.arange(0, 0.1, 1/fs)

# 基带信号 (QPSK)
I_base = np.random.choice([-1, 1], size=len(t)) / np.sqrt(2)
Q_base = np.random.choice([-1, 1], size=len(t)) / np.sqrt(2)

# 调制: 上变频
s_rf = I_base * np.cos(2*np.pi*fc*t) - Q_base * np.sin(2*np.pi*fc*t)

# 解调: 下变频
I_demod = s_rf * 2 * np.cos(2*np.pi*fc*t)  # 乘以同相载波
Q_demod = s_rf * (-2) * np.sin(2*np.pi*fc*t)  # 乘以正交载波

# 低通滤波 (去除2倍频分量)
b, a = signal.butter(6, 20, fs=fs)
I_recovered = signal.filtfilt(b, a, I_demod)
Q_recovered = signal.filtfilt(b, a, Q_demod)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(t[:100], I_base[:100], label='I baseband')
axes[0, 0].plot(t[:100], Q_base[:100], label='Q baseband')
axes[0, 0].set_title('Baseband Signal')
axes[0, 0].legend()

axes[0, 1].plot(t[:100], s_rf[:100])
axes[0, 1].set_title('Modulated RF Signal')

axes[1, 0].plot(t[:100], I_demod[:100], alpha=0.5, label='Before LPF')
axes[1, 0].plot(t[:100], I_recovered[:100], linewidth=2, label='After LPF')
axes[1, 0].set_title('I Channel Demodulation')
axes[1, 0].legend()

axes[1, 1].plot(t[:100], Q_demod[:100], alpha=0.5, label='Before LPF')
axes[1, 1].plot(t[:100], Q_recovered[:100], linewidth=2, label='After LPF')
axes[1, 1].set_title('Q Channel Demodulation')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("调制: I*cos + Q*sin → 上变频到射频")
print("解调: 乘以本地载波 + 低通滤波 → 恢复I/Q")
print("RFFI中的IQ数据就是解调后的基带信号")

In [ ]:
# IQ信号的关键参数
print("=== IQ信号的关键参数 ===")
print()
print("1. 幅度 (Amplitude):")
print("   A = sqrt(I^2 + Q^2)")
print("   反映信号强度")
print()
print("2. 相位 (Phase):")
print("   φ = arctan2(Q, I)")
print("   反映信号角度, 对载波频偏和相位偏移敏感")
print()
print("3. 瞬时频率 (Instantaneous Frequency):")
print("   f_inst = dφ/dt / (2π)")
print("   相位的导数, 反映频率随时间的变化")
print()
print("4. 功率 (Power):")
print("   P = I^2 + Q^2 = A^2")
print()
print("5. 信噪比 (SNR):")
print("   SNR = 10*log10(P_signal / P_noise)  [dB]")
print()
print("6. 误差向量幅度 (EVM):")
print("   EVM = sqrt(|r - r_ideal|^2) / |r_ideal| * 100%")
print("   衡量调制质量, RFFI中与指纹强度相关")

---
## 4. 滤波器设计

滤波器是信号处理的核心工具:
- 低通: 去除高频噪声
- 高通: 去除低频漂移
- 带通: 提取特定频段
- 带阻: 去除特定干扰

In [ ]:
# 滤波器类型对比
fs = 1000
cutoff = 100

b_butter, a_butter = signal.butter(6, cutoff, fs=fs)
b_cheby1, a_cheby1 = signal.cheby1(6, 1, cutoff, fs=fs)
b_ellip, a_ellip = signal.ellip(6, 1, 40, cutoff, fs=fs)
b_fir = signal.firwin(101, cutoff, fs=fs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 频率响应
for name, b, a in [('Butterworth', b_butter, a_butter),
                     ('Chebyshev I', b_cheby1, a_cheby1),
                     ('Elliptic', b_ellip, a_ellip)]:
    w, h = signal.freqz(b, a, fs=fs)
    axes[0].plot(w, 20*np.log10(np.abs(h)+1e-10), label=name)

w, h = signal.freqz(b_fir, fs=fs)
axes[0].plot(w, 20*np.log10(np.abs(h)+1e-10), label='FIR (101 taps)')
axes[0].set_title('Low-pass Filter Frequency Response')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Magnitude (dB)')
axes[0].legend()
axes[0].set_ylim(-80, 5)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(cutoff, color='k', linestyle='--', alpha=0.5)

# 滤波效果
t = np.arange(0, 0.5, 1/fs)
x_clean = np.sin(2*np.pi*30*t)
x_noise = x_clean + 0.5*np.sin(2*np.pi*200*t) + 0.3*np.random.randn(len(t))
x_filtered = signal.filtfilt(b_butter, a_butter, x_noise)

axes[1].plot(t[:200], x_noise[:200], alpha=0.4, label='Noisy')
axes[1].plot(t[:200], x_clean[:200], 'k--', label='Original')
axes[1].plot(t[:200], x_filtered[:200], linewidth=2, label='Filtered')
axes[1].set_title('Butterworth Low-pass Filter Effect')
axes[1].legend()
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

print("Butterworth: 最平坦通带, 过渡带较宽")
print("Chebyshev I: 通带有波纹, 过渡带陡峭")
print("Elliptic:    通带阻带都有波纹, 过渡带最陡")
print("FIR:         线性相位, 阶数高, 稳定")

In [ ]:
# IIR vs FIR
print("=== IIR vs FIR 滤波器 ===")
print()
print("IIR (无限脉冲响应):")
print("  y[n] = Σ b[k]*x[n-k] - Σ a[k]*y[n-k]")
print("  阶数低, 计算快, 可能不稳定, 非线性相位")
print("  例: Butterworth, Chebyshev, Elliptic")
print()
print("FIR (有限脉冲响应):")
print("  y[n] = Σ b[k]*x[n-k]")
print("  阶数高, 计算慢, 总是稳定, 线性相位")
print("  例: 窗函数法, Parks-McClellan")
print()
print("RFFI中常用:")
print("  接收端: 低通IIR滤波 (去除带外噪声)")
print("  预处理: 带通FIR滤波 (提取信号频段)")
print("  filtfilt: 零相位滤波 (正向+反向滤波, 无相位失真)")

---
## 5. 信道效应

信号从发射端到接收端经历的物理过程:

```
发射信号 → 路径损耗 → 多径衰落 → 多普勒 → 加性噪声 → 接收信号
```

这些信道效应会叠加在射频指纹上, 是RFFI泛化的主要障碍。

In [ ]:
# 1. 加性高斯白噪声 (AWGN)
np.random.seed(42)
fs = 1000
t = np.arange(0, 0.5, 1/fs)
x = np.sin(2*np.pi*50*t)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, snr_db in zip(axes, [30, 20, 10, 0]):
    signal_power = np.mean(x**2)
    noise_power = signal_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power) * np.random.randn(len(x))
    x_noisy = x + noise
    ax.plot(t[:200], x_noisy[:200])
    ax.set_title(f'SNR = {snr_db} dB')
    ax.set_xlabel('Time (s)')

plt.suptitle('AWGN Channel: Different SNR Levels', fontsize=13)
plt.tight_layout()
plt.show()
print("SNR越低, 噪声越大, 射频指纹越难提取")

In [ ]:
# 2. 载波频偏 (CFO - Carrier Frequency Offset)
fs = 1000
t = np.arange(0, 0.5, 1/fs)
I = np.ones(len(t)) / np.sqrt(2)
Q = np.ones(len(t)) / np.sqrt(2)
s = I + 1j * Q  # QPSK基带信号

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, cfo in zip(axes, [0, 5, 20, 50]):
    s_cfo = s * np.exp(1j * 2 * np.pi * cfo * t / fs)
    ax.scatter(s_cfo.real[::5], s_cfo.imag[::5], alpha=0.3, s=5)
    ax.set_title(f'CFO = {cfo} Hz')
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Carrier Frequency Offset Effect on Constellation', fontsize=13)
plt.tight_layout()
plt.show()
print("CFO使星座图旋转, 是RFFI中重要的指纹来源(振荡器偏差)")
print("但也是信道干扰, 需要区分指纹CFO和信道CFO")

In [ ]:
# 3. IQ不平衡 (IQ Imbalance)
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

n_pts = 500
I = np.random.choice([-1, 1], n_pts) / np.sqrt(2)
Q = np.random.choice([-1, 1], n_pts) / np.sqrt(2)

params = [
    ('No Imbalance', 1.0, 0),
    ('Amp Imbalance', 1.3, 0),
    ('Phase Imbalance', 1.0, 15),
    ('Both', 1.2, 10),
]

for ax, (name, amp_imb, phase_deg) in zip(axes, params):
    phase_rad = np.deg2rad(phase_deg)
    I_imb = I * amp_imb
    Q_imb = Q * np.cos(phase_rad) + I * np.sin(phase_rad)
    ax.scatter(I_imb, Q_imb, alpha=0.5)
    ax.set_title(name)
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.set_aspect('equal')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

plt.suptitle('IQ Imbalance Effect on Constellation', fontsize=13)
plt.tight_layout()
plt.show()
print("IQ不平衡是RFFI最重要的指纹来源之一")
print("由混频器的幅度和相位不匹配引起")

In [ ]:
# 4. 多径衰落
print("=== 多径衰落 ===")
print()
print("信号经多条路径到达接收端:")
print("  直射路径 (LOS): 最短, 最强")
print("  反射路径: 延迟, 衰减, 相位旋转")
print()
print("数学模型:")
print("  h(t) = Σ a_i * δ(t - τ_i) * exp(jφ_i)")
print("  a_i: 第i条路径的衰减")
print("  τ_i: 第i条路径的时延")
print("  φ_i: 第i条路径的相移")
print()
print("平坦衰落: 信号带宽 << 信道相干带宽")
print("  所有频率分量衰减相同")
print("频率选择性衰落: 信号带宽 > 信道相干带宽")
print("  不同频率分量衰减不同, 产生码间干扰")
print()
print("RFFI中: 多径是干扰, 会掩盖指纹特征")
print("  需要信道均衡或数据增强来对抗多径影响")

In [ ]:
# 5. 功率放大器非线性
t = np.linspace(-2, 2, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AM/AM 特性 (幅度非线性)
x_pa = np.linspace(0, 2, 100)
y_linear = x_pa
y_soft = x_pa / (1 + 0.3 * x_pa**2)  # Rapp模型简化
y_hard = np.tanh(x_pa)  # 饱和模型

axes[0].plot(x_pa, y_linear, 'k--', label='Linear')
axes[0].plot(x_pa, y_soft, 'b-', label='Soft Nonlinearity')
axes[0].plot(x_pa, y_hard, 'r-', label='Hard Nonlinearity')
axes[0].set_xlabel('Input Amplitude')
axes[0].set_ylabel('Output Amplitude')
axes[0].set_title('PA AM/AM Characteristic')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PA非线性对星座图的影响
n_pts = 500
I = np.random.choice([-1, 1], n_pts) / np.sqrt(2)
Q = np.random.choice([-1, 1], n_pts) / np.sqrt(2)
s = I + 1j * Q

# 不同PA非线性系数
for alpha, label in [(0, 'Linear'), (0.1, 'Mild'), (0.3, 'Strong')]:
    s_pa = s + alpha * s * np.abs(s)**2  # 三阶非线性
    axes[1].scatter(s_pa.real, s_pa.imag, alpha=0.5, label=label, s=10)

axes[1].set_title('PA Nonlinearity Effect on Constellation')
axes[1].set_xlabel('I')
axes[1].set_ylabel('Q')
axes[1].set_aspect('equal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("PA非线性使星座点向外扩散, 是RFFI的重要指纹来源")
print("不同设备的PA非线性系数不同, 形成独特的指纹")

---
## 6. 信号检测与参数估计

RFFI中常用的信号处理技术。

In [ ]:
# 1. 能量检测
np.random.seed(42)
fs = 1000
t = np.arange(0, 1, 1/fs)

signal_present = np.sin(2*np.pi*50*t) + 0.1*np.random.randn(len(t))
signal_absent = 0.1*np.random.randn(len(t))

# 计算能量
window_size = 50
energy_present = np.convolve(signal_present**2, np.ones(window_size)/window_size, mode='same')
energy_absent = np.convolve(signal_absent**2, np.ones(window_size)/window_size, mode='same')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(t, signal_present, alpha=0.5, label='Signal+Noise')
axes[0].plot(t, signal_absent, alpha=0.5, label='Noise Only')
axes[0].set_title('Received Signal')
axes[0].legend()

axes[1].plot(t, energy_present, label='Signal Present')
axes[1].plot(t, energy_absent, label='Signal Absent')
axes[1].axhline(0.3, color='r', linestyle='--', label='Threshold')
axes[1].set_title('Energy Detection')
axes[1].legend()

plt.tight_layout()
plt.show()
print("能量检测: 最简单的信号检测方法")
print("信号能量 > 阈值 → 判定信号存在")

In [ ]:
# 2. 自相关函数与周期性检测
fs = 1000
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*50*t) + 0.5*np.random.randn(len(t))

# 自相关
autocorr = np.correlate(x, x, mode='full')
autocorr = autocorr[len(x)-1:]  # 取正半部分
autocorr = autocorr / autocorr[0]  # 归一化

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(t[:200], x[:200])
axes[0].set_title('Noisy Sinusoidal Signal')
axes[0].set_xlabel('Time (s)')

lags = np.arange(0, 200)
axes[1].plot(lags/fs*1000, autocorr[:200])
axes[1].set_title('Autocorrelation Function')
axes[1].set_xlabel('Lag (ms)')
axes[1].axhline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# 找周期
peaks = signal.find_peaks(autocorr[:500], distance=10)[0]
if len(peaks) > 0:
    period_samples = peaks[0]
    detected_freq = fs / period_samples
    print(f"自相关检测到周期: {period_samples} 采样点")
print(f"检测频率: {detected_freq:.1f} Hz (真实: 50 Hz)")

In [ ]:
# 3. 循环平稳分析 (Cyclostationary)
print("=== 循环平稳分析 ===")
print()
print("调制信号具有循环平稳性:")
print("  统计特性随时间周期性变化")
print("  不同调制方式有不同的循环频率")
print()
print("循环自相关函数:")
print("  R_x(τ, α) = E[x(t) * x*(t-τ) * e^(-j2παt)]")
print("  α: 循环频率")
print()
print("应用:")
print("  调制识别: 不同调制有不同的循环频率")
print("  信号检测: 在噪声中检测微弱信号")
print("  RFFI: 循环平稳特征可能包含设备指纹信息")

In [ ]:
# 4. 功率谱密度估计
fs = 1000
t = np.arange(0, 5, 1/fs)
x = np.sin(2*np.pi*50*t) + 0.5*np.sin(2*np.pi*120*t) + np.random.randn(len(t))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 周期图法 (Periodogram)
f_per, Pxx_per = signal.periodogram(x, fs=fs)
axes[0].semilogy(f_per, Pxx_per)
axes[0].set_title('Periodogram')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('PSD')
axes[0].set_xlim(0, 200)

# Welch法 (改进的周期图)
f_welch, Pxx_welch = signal.welch(x, fs=fs, nperseg=512)
axes[1].semilogy(f_welch, Pxx_welch)
axes[1].set_title('Welch Method')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 200)

# 对比
axes[2].semilogy(f_per, Pxx_per, alpha=0.5, label='Periodogram')
axes[2].semilogy(f_welch, Pxx_welch, alpha=0.8, label='Welch')
axes[2].set_title('Comparison')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_xlim(0, 200)
axes[2].legend()

plt.tight_layout()
plt.show()
print("周期图法: 方差大, 不稳定")
print("Welch法: 分段平均, 方差小, 分辨率略低")
print("RFFI中常用Welch法估计PSD, 作为特征输入")

---
## 7. Python 信号处理实战 (scipy.signal)

In [ ]:
# scipy.signal 常用函数速查
print("=== scipy.signal 常用函数 ===")
print()
print("滤波:")
print("  signal.butter(N, Wn)      - Butterworth滤波器")
print("  signal.cheby1(N, rp, Wn)  - Chebyshev I型")
print("  signal.firwin(numtaps, Wn) - FIR滤波器")
print("  signal.lfilter(b, a, x)   - IIR滤波")
print("  signal.filtfilt(b, a, x)  - 零相位滤波")
print()
print("频谱分析:")
print("  signal.periodogram(x, fs)  - 周期图")
print("  signal.welch(x, fs)        - Welch PSD")
print("  signal.stft(x, fs)         - 短时傅里叶变换")
print("  signal.csd(x, y, fs)       - 互谱密度")
print()
print("信号生成:")
print("  signal.chirp(t, f0, f1, t1) - 线性调频")
print("  signal.square(t)            - 方波")
print("  signal.sawtooth(t)          - 锯齿波")
print()
print("峰值检测:")
print("  signal.find_peaks(x)        - 找峰值")
print()
print("重采样:")
print("  signal.resample(x, num)     - 重采样")
print("  signal.decimate(x, q)       - 降采样")
print("  signal.upfirdn(h, x, up, down) - 多相重采样")
print()
print("卷积:")
print("  np.convolve(x, h)           - 线性卷积")
print("  signal.fftconvolve(x, h)    - FFT卷积(长序列更快)")

In [ ]:
# RFFI信号处理完整流程
print("=== RFFI 信号处理完整流程 ===")
print()
print("1. 信号采集")
print("   SDR接收 → 下变频 → IQ采样")
print("   采样率: 通常 1~20 MSps")
print()
print("2. 预处理")
print("   a. 去除DC偏移: x = x - mean(x)")
print("   b. 归一化: x = x / std(x)")
print("   c. 低通滤波: 去除带外噪声")
print("   d. 重采样: 统一采样率")
print()
print("3. 特征提取 (传统方法)")
print("   a. 时域: 均值、方差、峰度、偏度")
print("   b. 频域: PSD、谱质心、谱熵")
print("   c. 时频域: STFT、小波变换")
print("   d. 高阶统计量: 高阶累积量")
print()
print("4. 深度学习方法")
print("   a. 原始IQ → 1D CNN (端到端, 最常用)")
print("   b. 时频图 → 2D CNN")
print("   c. IQ+手工特征 → 多模态融合")
print()
print("5. 后处理")
print("   a. 置信度阈值 (Open-set)")
print("   b. 多帧投票 (时序平滑)")
print("   c. 多接收机融合")

In [ ]:
# 完整的IQ信号处理示例
np.random.seed(42)
fs = 1e6  # 1 MSps
n_samples = 1024
fc = 0.05 * fs  # 载波 50kHz (基带偏移)

t = np.arange(n_samples) / fs

# 生成QPSK信号
I = np.random.choice([-1, 1], n_samples) / np.sqrt(2)
Q = np.random.choice([-1, 1], n_samples) / np.sqrt(2)
s = I + 1j * Q

# 添加设备指纹
cfo = 500  # 载波频偏 500Hz
s = s * np.exp(1j * 2 * np.pi * cfo * t)

amp_imb = 1.1
phase_imb = np.deg2rad(5)
I_imb = s.real * amp_imb
Q_imb = s.imag * np.cos(phase_imb) + s.real * np.sin(phase_imb)
s = I_imb + 1j * Q_imb

# PA非线性
s = s - 0.05 * s * np.abs(s)**2

# AWGN
snr_db = 15
noise_power = np.mean(np.abs(s)**2) / (10**(snr_db/10))
noise = np.sqrt(noise_power/2) * (np.random.randn(n_samples) + 1j*np.random.randn(n_samples))
r = s + noise

# 处理步骤
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. 原始IQ
axes[0, 0].plot(t[:200]*1e3, r.real[:200], label='I')
axes[0, 0].plot(t[:200]*1e3, r.imag[:200], label='Q')
axes[0, 0].set_title('1. Raw IQ Signal')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Time (ms)')

# 2. 去DC
r_dc = r - np.mean(r)
axes[0, 1].plot(t[:200]*1e3, r_dc.real[:200], label='I')
axes[0, 1].set_title('2. DC Removal')
axes[0, 1].set_xlabel('Time (ms)')

# 3. 归一化
r_norm = r_dc / np.std(r_dc)
axes[0, 2].plot(t[:200]*1e3, r_norm.real[:200], label='I')
axes[0, 2].set_title('3. Normalization')
axes[0, 2].set_xlabel('Time (ms)')

# 4. 星座图
axes[1, 0].scatter(r_norm.real[::4], r_norm.imag[::4], alpha=0.3, s=5)
axes[1, 0].set_title('4. Constellation')
axes[1, 0].set_xlabel('I')
axes[1, 0].set_ylabel('Q')
axes[1, 0].set_aspect('equal')

# 5. PSD
f_psd, Pxx = signal.welch(r_norm, fs=fs, nperseg=256, return_onesided=False)
f_psd = fftshift(f_psd)
Pxx = fftshift(Pxx)
axes[1, 1].semilogy(f_psd/1e3, Pxx)
axes[1, 1].set_title('5. Power Spectral Density')
axes[1, 1].set_xlabel('Frequency (kHz)')
axes[1, 1].set_ylabel('PSD')

# 6. 瞬时相位
phase = np.unwrap(np.angle(r_norm))
inst_freq = np.diff(phase) / (2*np.pi) * fs
axes[1, 2].plot(t[1:200]*1e3, inst_freq[:199])
axes[1, 2].set_title('6. Instantaneous Frequency')
axes[1, 2].set_xlabel('Time (ms)')
axes[1, 2].set_ylabel('Freq (Hz)')

plt.tight_layout()
plt.show()
print("这是RFFI中典型的IQ信号处理流程")

---
## 总结

### 信号处理知识图谱 (RFFI视角)

```
信号处理基础
├── 信号基础
│   ├── 采样定理: fs ≥ 2*fmax
│   ├── 混叠: 欠采样的后果
│   └── 信号类型: 正弦/方波/脉冲/噪声
│
├── 傅里叶变换
│   ├── DFT/FFT: 时域→频域
│   ├── 窗函数: 减少频谱泄漏
│   ├── STFT: 时频分析
│   └── PSD: 功率谱密度估计
│
├── IQ信号
│   ├── I/Q分量: 复基带表示
│   ├── 调制: BPSK/QPSK/QAM
│   ├── 解调: 下变频+低通
│   └── 关键参数: 幅度/相位/EVM
│
├── 滤波器
│   ├── IIR: Butterworth/Cheby/Ellip
│   ├── FIR: 线性相位
│   └── filtfilt: 零相位滤波
│
├── 信道效应
│   ├── AWGN: 加性噪声
│   ├── CFO: 载波频偏 (指纹来源!)
│   ├── IQ不平衡: 混频器缺陷 (指纹来源!)
│   ├── PA非线性: 功放失真 (指纹来源!)
│   └── 多径衰落: 信道干扰
│
└── 参数估计
    ├── 能量检测
    ├── 自相关/周期检测
    ├── 循环平稳分析
    └── PSD估计
```

### RFFI中信号处理的核心作用

| 信号处理技术 | 在RFFI中的作用 |
|------------|---------------|
| IQ归一化 | 标准化输入, 消除幅度差异 |
| 低通滤波 | 去除带外噪声 |
| FFT/PSD | 频域特征提取 |
| STFT | 时频图输入 (2D CNN) |
| CFO估计 | 区分指纹CFO和信道CFO |
| 信道均衡 | 消除多径对指纹的干扰 |
| 数据增强 | 模拟信道变化 |